# Final Evaluation — Two-Stage K-Means

Evaluates the frozen two-stage K-Means configuration on the untouched **9 final-evaluation images**.

The configuration is loaded from `recommended_core_parameters.csv`, which should be regenerated by the updated `parameter_tuning.ipynb` after removing Rule A/B. No retuning is performed on the final-evaluation split.


In [5]:
from pathlib import Path
from dataclasses import replace
from itertools import product
import random
import sys

import cv2
import numpy as np
import pandas as pd


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in [start, *start.parents]:
        if (
            (candidate / "banana_quality" / "config.py").exists()
            and (candidate / "data" / "annotations").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not find project root. Start Jupyter from inside the banana project."
    )

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from banana_quality.config import ProcessingConfig
from banana_quality.segmentation.kmeans import prepare_image
from banana_quality.segmentation.mask_refinement import banana_mask_from_prepared
from banana_quality.blemish.detection import blemish_mask_from_prepared
from banana_quality.surface_quality import blemish_percentage

DATASET_ROOT = PROJECT_ROOT / "data" / "annotations"
RESULTS_DIR = PROJECT_ROOT / "evaluation" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
SPLIT_SEED = 42
FINAL_EVALUATION_COUNT = 9

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)


Project root: c:\Users\enton\Project\python\banana_quality_ripeness_assessment_system
Dataset root: c:\Users\enton\Project\python\banana_quality_ripeness_assessment_system\data\annotations


In [6]:
images_dir = DATASET_ROOT / "images"
image_paths = sorted(
    p for p in images_dir.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

shuffled_paths = image_paths.copy()
rng = random.Random(SPLIT_SEED)
rng.shuffle(shuffled_paths)
final_names = {p.name for p in shuffled_paths[:FINAL_EVALUATION_COUNT]}

split_df = pd.DataFrame([
    {
        "image": p.name,
        "split": "final_evaluation" if p.name in final_names else "tuning",
    }
    for p in image_paths
]).sort_values(["split", "image"]).reset_index(drop=True)

split_df.to_csv(RESULTS_DIR / "dataset_split.csv", index=False)

assert len(image_paths) == 30, f"Expected 30 images, found {len(image_paths)}"
assert int((split_df["split"] == "tuning").sum()) == 21
assert int((split_df["split"] == "final_evaluation").sum()) == 9

print(split_df["split"].value_counts())


split
tuning              21
final_evaluation     9
Name: count, dtype: int64


In [7]:
def matching_mask(folder: Path, image_path: Path) -> Path:
    path = folder / f"{image_path.stem}.png"
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def load_binary_mask(path: Path, target_shape: tuple[int, int]) -> np.ndarray:
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f"Cannot read mask: {path}")
    mask = np.where(mask > 0, 255, 0).astype(np.uint8)
    if mask.shape != target_shape:
        h, w = target_shape
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
    return mask


def resize_binary_mask(mask: np.ndarray, target_shape: tuple[int, int]) -> np.ndarray:
    h, w = target_shape
    resized = cv2.resize(
        np.asarray(mask, dtype=np.uint8),
        (w, h),
        interpolation=cv2.INTER_NEAREST,
    )
    return np.where(resized > 0, 255, 0).astype(np.uint8)


def iou_score(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    pred = prediction > 0
    gt = ground_truth > 0
    union = int(np.logical_or(pred, gt).sum())
    if union == 0:
        return 1.0
    return float(np.logical_and(pred, gt).sum() / union)


def dice_score(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    pred = prediction > 0
    gt = ground_truth > 0
    denominator = int(pred.sum() + gt.sum())
    if denominator == 0:
        return 1.0
    return float(2.0 * np.logical_and(pred, gt).sum() / denominator)


def load_samples(split_name: str):
    names = set(split_df.loc[split_df["split"] == split_name, "image"])
    samples = []
    for image_path in image_paths:
        if image_path.name not in names:
            continue
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            raise ValueError(f"Cannot read image: {image_path}")
        shape = image.shape[:2]
        banana_gt = load_binary_mask(
            matching_mask(DATASET_ROOT / "banana_masks", image_path),
            shape,
        )
        blemish_gt = load_binary_mask(
            matching_mask(DATASET_ROOT / "blemish_masks", image_path),
            shape,
        )
        samples.append({
            "name": image_path.name,
            "image": image,
            "shape": shape,
            "banana_gt": banana_gt,
            "blemish_gt": blemish_gt,
        })
    return samples


## 1. Load the tuned two-stage K-Means parameters

The banana-segmentation and blemish-detection parameters come directly from the updated parameter-tuning notebook. The final detector contains no supplemental Rule A/B stage.


In [8]:
core_path = RESULTS_DIR / "recommended_core_parameters.csv"
if not core_path.exists():
    raise FileNotFoundError(
        "Run parameter_tuning.ipynb first. Missing: " + str(core_path)
    )

core = pd.read_csv(core_path).iloc[0]

BANANA_K = int(core["kmeans_k"])
BANANA_ACCEPTANCE = float(core["banana_cluster_score_threshold"])
BANANA_OPENING = int(core["banana_opening_kernel"])
BANANA_CLOSING = int(core["banana_closing_kernel"])
BLEMISH_K = int(core["blemish_kmeans_k"])
BLEMISH_CHROMA = float(core["blemish_cluster_chroma_threshold"])
BLEMISH_OPENING = int(core["blemish_opening_kernel"])
BLEMISH_CLOSING = int(core["blemish_closing_kernel"])

print("Loaded tuned two-stage K-Means parameters:")
display(pd.DataFrame([{
    "banana_k": BANANA_K,
    "banana_acceptance": BANANA_ACCEPTANCE,
    "banana_opening": BANANA_OPENING,
    "banana_closing": BANANA_CLOSING,
    "blemish_k": BLEMISH_K,
    "blemish_chroma": BLEMISH_CHROMA,
    "blemish_opening": BLEMISH_OPENING,
    "blemish_closing": BLEMISH_CLOSING,
}]))

DEFAULTS = ProcessingConfig()
final_config = replace(
    DEFAULTS,
    kmeans_k=BANANA_K,
    banana_cluster_score_threshold=BANANA_ACCEPTANCE,
    blemish_kmeans_k=BLEMISH_K,
    blemish_cluster_chroma_threshold=BLEMISH_CHROMA,
)


Loaded tuned two-stage K-Means parameters:


,banana_k,banana_acceptance,banana_opening,banana_closing,blemish_k,blemish_chroma,blemish_opening,blemish_closing
0,7,0.35,3,7,6,21.0,3,5


## 2. Evaluate only the 9 untouched images


In [9]:
final_samples = load_samples("final_evaluation")
assert len(final_samples) == 9
rows = []
for sample in final_samples:
    prepared = prepare_image(sample["image"], final_config)
    banana_steps = banana_mask_from_prepared(
        prepared,
        opening_kernel_size=int(BANANA_OPENING),
        closing_kernel_size=int(BANANA_CLOSING),
    )
    banana_working = np.asarray(banana_steps["banana_mask"], dtype=np.uint8)
    banana_pred = resize_binary_mask(banana_working, sample["shape"])

    blemish_steps = blemish_mask_from_prepared(
        prepared,
        banana_working,
        opening_kernel_size=int(BLEMISH_OPENING),
        closing_kernel_size=int(BLEMISH_CLOSING),
        config=final_config,
    )
    blemish_pred = resize_binary_mask(
        blemish_steps["blemish_mask"],
        sample["shape"],
    )

    gt_pct = blemish_percentage(sample["banana_gt"], sample["blemish_gt"])
    pred_pct = blemish_percentage(banana_pred, blemish_pred)
    rows.append({
        "image": sample["name"],
        "banana_iou": iou_score(banana_pred, sample["banana_gt"]),
        "banana_dice": dice_score(banana_pred, sample["banana_gt"]),
        "blemish_iou": iou_score(blemish_pred, sample["blemish_gt"]),
        "blemish_dice": dice_score(blemish_pred, sample["blemish_gt"]),
        "ground_truth_blemish_pct": gt_pct,
        "predicted_blemish_pct": pred_pct,
        "blemish_percentage_error": abs(pred_pct - gt_pct),
    })

final_results = pd.DataFrame(rows)
final_results.to_csv(RESULTS_DIR / "final_evaluation_per_image.csv", index=False)
display(final_results)
print("Saved final_evaluation_per_image.csv")


,image,banana_iou,banana_dice,blemish_iou,blemish_dice,ground_truth_blemish_pct,predicted_blemish_pct,blemish_percentage_error
0,musa-acuminata-banana-8e3c133d-394a-11ec-bc37-...,0.885436,0.939237,0.466759,0.636449,40.541400,36.199865,4.341535
1,musa-acuminata-banana-8f7de794-394a-11ec-a8cc-...,0.864546,0.927353,0.440727,0.611812,39.844137,23.714242,16.129895
2,musa-acuminata-banana-afe6ec6d-394a-11ec-bcab-...,0.976443,0.988081,0.000000,0.000000,3.907305,0.000000,3.907305
3,musa-acuminata-banana-b01b3700-394a-11ec-b4a6-...,0.975709,0.987705,0.288381,0.447664,5.732035,1.634954,4.097081
4,musa-acuminata-banana-b01d2f7c-394a-11ec-a466-...,0.974754,0.987216,0.351752,0.520439,8.439791,3.424506,5.015285
5,musa-acuminata-banana-b2a5cf68-394a-11ec-b7c7-...,0.988356,0.994144,0.180756,0.306170,8.059516,1.480599,6.578917
6,musa-acuminata-freshunripe-1da23724-2653-11ec-...,0.902149,0.948558,0.000000,0.000000,2.812789,0.000000,2.812789
7,musa-acuminata-freshunripe-1fd8dd9a-2653-11ec-...,0.970623,0.985092,0.358842,0.528159,2.666086,4.292279,1.626193
8,musa-acuminata-freshunripe-2cdf79be-2653-11ec-...,0.842397,0.914458,0.000000,0.000000,0.000000,13.732288,13.732288


Saved final_evaluation_per_image.csv


## 3. Final metric summary


In [10]:
final_summary = pd.DataFrame([{
    "images": int(len(final_results)),
    "banana_mean_iou": float(final_results["banana_iou"].mean()),
    "banana_mean_dice": float(final_results["banana_dice"].mean()),
    "blemish_mean_iou": float(final_results["blemish_iou"].mean()),
    "blemish_mean_dice": float(final_results["blemish_dice"].mean()),
    "blemish_percentage_mae": float(final_results["blemish_percentage_error"].mean()),
}])
final_summary.to_csv(RESULTS_DIR / "final_evaluation_summary.csv", index=False)
display(final_summary)
print("Saved final_evaluation_summary.csv")


,images,banana_mean_iou,banana_mean_dice,blemish_mean_iou,blemish_mean_dice,blemish_percentage_mae
0,9,0.931157,0.963538,0.231913,0.338966,6.471254


Saved final_evaluation_summary.csv


## End of Notebook
These are reporting outputs only. The notebook does not interpret whether the result is good enough and does not modify any parameters.
